# ✋ Hand AI – ASL A-Z Training (26 classes)
Train a MobileNetV2 model on your ASL dataset from Google Drive.

**Output**: `hand_model.keras` with 26 output classes (A–Z)

### Steps:
1. Run cell 1 — mount your Google Drive
2. Set `TRAIN_DIR` to where your dataset lives on Drive
3. Run all remaining cells
4. Download `hand_model.keras` + `classes.json` at the end

In [ ]:
# ── Step 1: Mount Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Set this to your dataset folder on Drive ───────────────────────────────
# It should be the folder that contains A/, B/, C/, ... Z/ subfolders
# Example: '/content/drive/MyDrive/asl_alphabet_train'
TRAIN_DIR = '/content/drive/MyDrive/asl_alphabet_train'  # <-- CHANGE THIS

# Verify the path
if os.path.exists(TRAIN_DIR):
    folders = sorted(os.listdir(TRAIN_DIR))
    print(f'✅ Found {len(folders)} folders in dataset:')
    print(folders)
else:
    print(f'❌ Path not found: {TRAIN_DIR}')
    print('Check your Drive folder structure and update TRAIN_DIR above.')

In [ ]:
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print('TensorFlow version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
IMG_SIZE    = 224
BATCH       = 32
EPOCHS      = 10
NUM_CLASSES = 26   # A–Z only (skip del/nothing/space)

# Only A-Z (26 letters)
CLASSES = list('ABCDEFGHIJKLMNOPQRSTUVWXYZ')

print(f'Classes ({len(CLASSES)}):', CLASSES)

In [ ]:
# ── Data Generators ────────────────────────────────────────────────────────
datagen = ImageDataGenerator(
    rescale=1./127.5,
    preprocessing_function=lambda x: x - 1.0,  # normalize to [-1, 1] for MobileNetV2
    validation_split=0.1,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=False,  # ASL signs are hand-orientation specific — don't flip
)

train_gen = datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode='categorical',
    subset='training',
    classes=CLASSES,
)

val_gen = datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode='categorical',
    subset='validation',
    classes=CLASSES,
)

print('Class indices:', train_gen.class_indices)
print('Training samples:', train_gen.samples)
print('Validation samples:', val_gen.samples)

In [ ]:
# ── Save classes.json — MUST match model output order ─────────────────────
idx_to_class  = {v: k for k, v in train_gen.class_indices.items()}
classes_list  = [idx_to_class[i] for i in range(len(idx_to_class))]

with open('classes.json', 'w') as f:
    json.dump(classes_list, f)

print('classes.json saved:', classes_list)

In [ ]:
# ── Build Model ────────────────────────────────────────────────────────────
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax'),  # 26 classes
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()

In [ ]:
# ── Phase 1: Train classification head ────────────────────────────────────
print('Phase 1: Training head...')
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5, verbose=1),
    ]
)
print(f'Phase 1 done — best val accuracy: {max(history.history["val_accuracy"]):.2%}')

In [ ]:
# ── Phase 2: Fine-tune top layers for better accuracy ─────────────────────
print('Phase 2: Fine-tuning...')
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=5,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    ]
)
print(f'Phase 2 done — best val accuracy: {max(history2.history["val_accuracy"]):.2%}')

In [ ]:
# ── Save & Download ────────────────────────────────────────────────────────
model.save('hand_model.keras')
print('✅ Model saved as hand_model.keras')
print(f'   Output classes: {NUM_CLASSES}')

# Option A: Download directly to your computer
from google.colab import files
files.download('hand_model.keras')
files.download('classes.json')

# Option B: Save back to Drive (uncomment to use instead)
# import shutil
# shutil.copy('hand_model.keras', '/content/drive/MyDrive/hand_model.keras')
# shutil.copy('classes.json', '/content/drive/MyDrive/classes.json')
# print('Saved to Google Drive!')

## ✅ After downloading:
1. Copy `hand_model.keras` → `ai_models/hand-ai/saved_models/hand_model.keras` (replace existing)
2. Copy `classes.json` → `ai_models/hand-ai/saved_models/classes.json` (replace existing)
3. Restart `npm run dev` — the server will auto-load the 26-class model!

## 📁 Expected Drive folder structure:
```
MyDrive/
└── asl_alphabet_train/      ← set as TRAIN_DIR
    ├── A/
    │   ├── A1.jpg
    │   └── ...
    ├── B/
    ├── C/
    ├── ...
    └── Z/
```